# Time Series Components and Decomposition: Complete Guide

## 1. The Philosophy of Decomposition: The Symphony Analogy
- To understand decomposition, imagine a symphony. A raw time series is the combined audio of a hundred instruments playing at once. It sounds complex and hard to follow.
- Decomposition is the process of using an **equalizer** to isolate the individual sections:
  - The deep, steady beat of the percussion represents the **Trend**.
  - The recurring, predictable melody represents the **Seasonality**.
  - The unpredictable background chatter of the audience represents the **Noise** (Irregularity).

## 2. Why Decomposition is the Secret Weapon of Analysts
- **Forecasting Accuracy:** It is easier to forecast a stable trend and a repeating seasonal pattern separately than a noisy total.
- **Strategic Clarity:** Allows leadership to separate a long-term growth trend from a temporary seasonal dip.
- **Anomaly Detection:** By stripping away trend and seasonality, the remaining "noise" highlights actual business anomalies.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf

# Set display options and plot style
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for reproducibility
np.random.seed(42)
print("Libraries successfully imported and environment ready!")

## 3. Data Creation: Additive vs. Multiplicative Dynamics
- We will generate two distinct synthetic datasets representing Monthly Pharmaceutical Sales over 10 years (120 months).
- **Dataset 1 (Additive):** Represents a steady flu vaccine. The seasonal spike is a fixed number of units (e.g., +500 units in Winter), regardless of the baseline trend.
- **Dataset 2 (Multiplicative):** Represents a blockbuster lifestyle drug. The seasonal fluctuation is a percentage of the trend (e.g., +20% in Winter). As the trend grows, the absolute size of the seasonal spike grows proportionally.

In [ ]:
# Generate temporal index (120 months)
months = 120
date_rng = pd.date_range(start='2015-01-01', periods=months, freq='ME')
time = np.arange(months)

# Common Trend (Linear Growth)
base_sales = 1000
trend = base_sales + (10 * time)

# --- Additive Components ---
# Constant amplitude sine wave (+/- 300 units)
seasonal_add = 300 * np.sin(2 * np.pi * time / 12)
noise_add = np.random.normal(0, 80, months)
sales_additive = trend + seasonal_add + noise_add

# --- Multiplicative Components ---
# Seasonal factor fluctuates between 0.7 and 1.3 (+/- 30%)
seasonal_mult = 1 + 0.3 * np.sin(2 * np.pi * time / 12)
# Noise factor fluctuates tightly around 1.0
noise_mult = np.random.normal(1, 0.05, months)
sales_multiplicative = trend * seasonal_mult * noise_mult

# Create DataFrames
df_pharma = pd.DataFrame({
    'Date': date_rng,
    'Trend': trend,
    'Sales_Additive': sales_additive,
    'Sales_Multiplicative': sales_multiplicative
})
df_pharma.set_index('Date', inplace=True)

print("Synthetic Additive and Multiplicative datasets generated.")
print(df_pharma.head())

## 4. Visual Inspection: The Eyeball Test
- The first step in any decomposition process is a visual diagnostic.
- We look specifically at the **amplitude** of the seasonal cycles relative to the **level** of the trend.
- **Stable Amplitude:** Additive model.
- **Proportional Amplitude (Fan-out effect):** Multiplicative model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Plot Additive
axes[0].plot(df_pharma.index, df_pharma['Sales_Additive'], color='blue', label='Additive Sales')
axes[0].plot(df_pharma.index, df_pharma['Trend'], color='black', linestyle='--', label='Underlying Trend')
axes[0].set_title('Additive Model: Stable Amplitude', fontsize=14)
axes[0].legend()

# Plot Multiplicative
axes[1].plot(df_pharma.index, df_pharma['Sales_Multiplicative'], color='darkorange', label='Multiplicative Sales')
axes[1].plot(df_pharma.index, df_pharma['Trend'], color='black', linestyle='--', label='Underlying Trend')
axes[1].set_title('Multiplicative Model: Growing Amplitude (Fan-Out)', fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.show()

print("Notice how the peaks and troughs on the right (orange) grow wider apart as the trend increases.")

## 5. The Building Blocks: Trend, Seasonality, Noise
Decomposition effectively breaks the signal down into distinct mathematical categories:

- **Trend-Cycle (T_t):** Captures the long-term movement. Is the business fundamentally growing or shrinking?
- **Seasonality (S_t):** Captures predictable, recurring patterns over a fixed interval (e.g., 12 months).
- **Irregularity/Noise (I_t):** The residual truth. What remains after Trend and Seasonality are removed. A perfect model leaves only "white noise" where I_t ~ N(0, sigma^2).

In [ ]:
# We can plot the exact components we generated for the Additive model to visualize perfect isolation.
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(df_pharma.index, df_pharma['Trend'], color='black')
axes[0].set_title('True Trend (T_t)')

axes[1].plot(df_pharma.index, seasonal_add, color='green')
axes[1].set_title('True Seasonality (S_t)')

axes[2].plot(df_pharma.index, noise_add, color='red', marker='.', linestyle='None')
axes[2].set_title('True Irregular Noise (I_t)')

plt.tight_layout()
plt.show()

## 6. Classical Decomposition: The Additive Framework
- Formula: Y_t = T_t + S_t + C_t + I_t
- Use this when the amplitude of seasonal fluctuations does not change as the trend increases.
- We use `statsmodels.tsa.seasonal.seasonal_decompose` to extract these mathematically.

In [ ]:
# Perform Additive Decomposition
decompose_add = seasonal_decompose(df_pharma['Sales_Additive'], model='additive', period=12)

fig = decompose_add.plot()
fig.set_size_inches(12, 10)
fig.axes[0].set_title('Additive Decomposition Results', fontsize=16)
plt.tight_layout()
plt.show()

## 7. Classical Decomposition: The Multiplicative Framework
- Formula: Y_t = T_t * S_t * C_t * I_t
- Use this when the seasonal fluctuations scale proportionally with the trend. This is the gold standard for most economic and retail data.
- Let's apply multiplicative decomposition to our "fan-out" dataset.

In [ ]:
# Perform Multiplicative Decomposition
decompose_mult = seasonal_decompose(df_pharma['Sales_Multiplicative'], model='multiplicative', period=12)

fig = decompose_mult.plot()
fig.set_size_inches(12, 10)
fig.axes[0].set_title('Multiplicative Decomposition Results', fontsize=16)
plt.tight_layout()
plt.show()

## 8. Model Selection: What Happens When You Choose Wrong?
- Choosing between Additive and Multiplicative is a fundamental assertion about the mechanics of your data.
- If you apply an Additive model to Multiplicative data, the algorithm will fail to extract the scaling seasonality properly. 
- The leftover, unexplained variance will "leak" into the Residuals (Noise) component.

In [ ]:
# WRONG APPLICATION: Applying Additive Decomposition to Multiplicative Data
wrong_decompose = seasonal_decompose(df_pharma['Sales_Multiplicative'], model='additive', period=12)

plt.figure(figsize=(12, 4))
plt.plot(wrong_decompose.resid.index, wrong_decompose.resid, color='red', marker='.')
plt.title('Residuals of WRONG Model (Additive applied to Multiplicative)', fontsize=14)
plt.axhline(0, color='black', linestyle='--')
plt.tight_layout()
plt.show()

print("DIAGNOSTIC WARNING:")
print("Notice the 'bowtie' or 'fan' shape in the residuals. They start small and grow larger over time.")
print("This proves the residuals are NOT random white noise. The Additive model failed to capture the growing seasonal amplitude.")

## 9. The Log-Transformation Bridge
- What if your statistical package only supports Additive models? Or what if your Multiplicative data has extreme variance?
- You can use a log-transformation to turn a multiplicative relationship into an additive one:
- log(Y_t) = log(T_t * S_t * I_t) = log(T_t) + log(S_t) + log(I_t)
- By taking the natural logarithm, we compress large values, stabilize the variance, and apply additive decomposition perfectly.

In [ ]:
# Step 1: Log-transform the multiplicative data
df_pharma['Log_Sales'] = np.log(df_pharma['Sales_Multiplicative'])

# Step 2: Apply Additive Decomposition to the Log Data
log_decompose = seasonal_decompose(df_pharma['Log_Sales'], model='additive', period=12)

plt.figure(figsize=(12, 4))
plt.plot(log_decompose.resid.index, log_decompose.resid, color='green', marker='.')
plt.title('Residuals AFTER Log Transformation (Additive Model)', fontsize=14)
plt.axhline(0, color='black', linestyle='--')
plt.tight_layout()
plt.show()

print("SUCCESS: The fan-out effect is gone. The residuals now look like stable, random white noise.")
print("This demonstrates how Log-Transformation acts as a mathematical bridge between the two frameworks.")

## 10. Residual Diagnostics: Autocorrelation (ACF)
- The goal of modeling is to leave behind only white noise.
- If your residuals contain patterns, your model is failing to capture information.
- We use the Autocorrelation Function (ACF) to test if residuals are correlated with their own past values. If they are, the model is incomplete.
- First, let's look at the ACF of the BAD residuals (Additive applied to Multiplicative).

In [ ]:
# Drop NaNs created by the moving averages in decomposition
bad_residuals = wrong_decompose.resid.dropna()

fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(bad_residuals, ax=ax, lags=36, alpha=0.05, title="ACF of BAD Residuals (Significant Correlation)")
plt.show()

print("The blue shaded area represents the confidence interval.")
print("Because the spikes breach the blue area regularly, we have significant autocorrelation left over.")

## 11. Residual Diagnostics: Validating the Good Model
- Now, let's look at the ACF of the GOOD residuals (Multiplicative model on Multiplicative data).
- We expect almost no spikes to breach the blue confidence interval, proving it is pure white noise.

In [ ]:
good_residuals = decompose_mult.resid.dropna()

fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(good_residuals, ax=ax, lags=36, alpha=0.05, title="ACF of GOOD Residuals (White Noise)")
plt.show()

print("Excellent! The spikes are almost entirely contained within the blue threshold.")
print("This confirms our model successfully extracted all structural information (Trend + Seasonality).")

## 12. Strategic Application: Anomaly Detection
- If you remove the trend and the seasonality from your data, you are left with the Irregular Component (I_t).
- If that residual ever spikes significantly, you have effectively found an anomaly that requires immediate business investigation (e.g., a supply chain shock).

In [ ]:
# Inject an artificial anomaly (e.g., severe shortage in month 70)
df_anomaly = df_pharma.copy()
anomaly_index = df_anomaly.index[70]
df_anomaly.loc[anomaly_index, 'Sales_Additive'] -= 600

# Decompose the anomalous data
anomaly_decompose = seasonal_decompose(df_anomaly['Sales_Additive'], model='additive', period=12)

plt.figure(figsize=(12, 4))
plt.plot(anomaly_decompose.resid.index, anomaly_decompose.resid, color='gray')
plt.scatter(anomaly_index, anomaly_decompose.resid.loc[anomaly_index], color='red', s=100, zorder=5, label='Detected Anomaly')
plt.axhline(0, color='black', linestyle='--')
plt.title('Business Application: Anomaly Detection in the Residuals', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

print(f"The extreme negative spike in the residuals clearly isolates the event on {anomaly_index.date()}.")

## 13. Practice Exercises
**Scenario:** You are analyzing web traffic for an e-commerce platform over 4 years (48 months).
Let's generate a mystery dataset. Your task will be to analyze it.

In [ ]:
# Generate practice data
dates_web = pd.date_range('2021-01-01', periods=48, freq='ME')
t_web = np.arange(48)

web_trend = 5000 * np.exp(0.05 * t_web)
web_season = 1 + 0.4 * np.sin(2 * np.pi * t_web / 12)
web_noise = np.random.normal(1, 0.05, 48)

df_web = pd.DataFrame({
    'Traffic': web_trend * web_season * web_noise
}, index=dates_web)

print("Practice Web Traffic Dataset Created.")
print(df_web.head())

### Exercise 1: Identify the Model Type
- **Task:** Plot the raw Web Traffic data. Based on visual inspection (the Eyeball Test), decide whether you should use an Additive or Multiplicative model.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
plt.figure(figsize=(10, 4))
plt.plot(df_web.index, df_web['Traffic'], color='purple', marker='o')
plt.title('Web Traffic Over 4 Years', fontsize=14)
plt.ylabel('Visits')
plt.grid(True, alpha=0.3)
plt.show()

print("Answer: The amplitude of the seasonal swings grows massively as the traffic grows.")
print("This is a classic 'Fan-Out' effect. We MUST use a Multiplicative model.")

### Exercise 2: Log Transformation Execution
- **Task:** The engineering team's forecasting tool only accepts Additive data inputs. 
- Transform the `df_web` data using the Log-Transformation Bridge.
- Run an Additive decomposition on the log data and plot the results.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
df_web['Log_Traffic'] = np.log(df_web['Traffic'])

web_log_decompose = seasonal_decompose(df_web['Log_Traffic'], model='additive', period=12)

fig = web_log_decompose.plot()
fig.set_size_inches(12, 8)
fig.axes[0].set_title('Exercise 2: Additive Decomposition of Log-Transformed Web Traffic', fontsize=14)
plt.tight_layout()
plt.show()

print("Notice how the Trend is now perfectly linear (log of an exponential trend is linear),")
print("and the seasonal amplitude is perfectly constant. Transformation successful.")

## 14. Visualization Gallery: The Symphony Deconstructed
- Let's create one final, comprehensive visualization showing the step-by-step extraction of signals using the IQVIA-style pharmaceutical multiplicative model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(df_pharma.index, df_pharma['Sales_Multiplicative'], color='darkorange')
axes[0, 0].set_title('1. Raw Sales Data (The Full Symphony)', fontsize=14)

axes[0, 1].plot(df_pharma.index, decompose_mult.trend, color='black', linewidth=3)
axes[0, 1].set_title('2. Extracted Trend (The Percussion)', fontsize=14)

axes[1, 0].plot(df_pharma.index, decompose_mult.seasonal, color='green')
axes[1, 0].set_title('3. Isolated Seasonality (The Melody)', fontsize=14)

axes[1, 1].plot(df_pharma.index, decompose_mult.resid, color='red', marker='.', linestyle='None')
axes[1, 1].axhline(1, color='black', linestyle='--')
axes[1, 1].set_title('4. Remaining Irregularity (The Audience Noise)', fontsize=14)

for ax in axes.flatten():
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 15. Summary and Theoretical Takeaways

1. **Decomposition Philosophy**: Time series data is a composite of Trend, Seasonality, and Noise. Separating them decouples decision-making and clarifies strategy.
2. **Additive Models**: Y_t = T_t + S_t + I_t. Used when seasonal peaks maintain a constant absolute volume regardless of trend.
3. **Multiplicative Models**: Y_t = T_t * S_t * I_t. Used when seasonal peaks scale proportionally as a percentage of the growing trend (common in economics and pharma).

In [ ]:
print("Theoretical concepts successfully reviewed.")

## 16. Summary and Practical Takeaways

4. **Log-Transformation**: The natural log acts as a mathematical bridge, stabilizing variance and turning multiplicative relationships into additive ones (log(Y_t) = log(T_t) + log(S_t) + log(I_t)).
5. **Residual Diagnostics**: A successful decomposition leaves behind pure white noise. Use the ACF plot to ensure no patterns remain in the residuals.
6. **Anomaly Detection**: By monitoring the isolated Irregular/Noise component, businesses can automatically trigger alerts for sudden supply chain shocks or irregular demand.

In [ ]:
print("Module 'Time Series Components and Decomposition' completed successfully.")
print("You are now ready to rebuild these components into forward-looking forecasting models!")